# 05 Dodatkowe analizy – walidacja doboru modelu H4b (2016 vs 2024)

## 1. Konfiguracja

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC
from tqdm.auto import tqdm

from src.config import CAMP_TERMS, H4B_PHASES, H4B_PER_CELL, POLITICAL_SUBREDDITS, PROCESSED_DIR, RESULTS_DIR
from src.terms import mask_terms

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECK_CYCLES = [2016, 2024]
REPEATS = 100
SHUFFLE_REPEATS = 20
LEARNING_REPEATS = 20
LEARNING_SIZES = [1000, 2000, 4000, 6000, 8000]
CELLS = ["subreddit", "phase"]

## 2. Dane po maskowaniu

In [ ]:
assert any(CAMP_TERMS[camp][term_type] for camp in CAMP_TERMS for term_type in CAMP_TERMS[camp]), "CAMP_TERMS jest pusty"
sample = pd.read_parquet(PROCESSED_DIR / "sample.parquet", columns=["id", "link_id", "subreddit", "camp", "cycle", "phase", "text"])
data = sample[sample["subreddit"].isin(POLITICAL_SUBREDDITS) & sample["phase"].isin(H4B_PHASES) & sample["cycle"].isin(CHECK_CYCLES)]
data = data.reset_index(drop=True)
data["text"] = mask_terms(data["text"], CAMP_TERMS)
cell_sizes = data.groupby(["cycle", "subreddit", "phase"]).size().unstack("cycle")
cell_sizes

## 3. Wspólna liczebność w obu cyklach

In [ ]:
per_cell = H4B_PER_CELL or int(0.8 * cell_sizes.min().min())
print("komentarzy na komórkę:", per_cell, "| na obóz:", 4 * per_cell)

## 4. Modele (ten sam TF-IDF; model główny ustalony z góry: regresja logistyczna)

In [ ]:
classifiers = {"regresja logistyczna (główny)": LogisticRegression(max_iter=1000),
               "naiwny Bayes": MultinomialNB(),
               "liniowy SVM": LinearSVC()}
models = {name: make_pipeline(TfidfVectorizer(stop_words="english", min_df=5), classifier) for name, classifier in classifiers.items()}
main_model = models["regresja logistyczna (główny)"]

## 5. Powtórzenia: trafność każdego modelu w obu cyklach

In [ ]:
rows = []
for repetition in tqdm(range(REPEATS)):
    for cycle in CHECK_CYCLES:
        part = data[data["cycle"] == cycle].groupby(CELLS).sample(per_cell, random_state=repetition)
        for name, model in models.items():
            predicted = cross_val_predict(model, part["text"], part["camp"], cv=GroupKFold(n_splits=5), groups=part["link_id"], n_jobs=-1)
            rows.append({"model": name, "cycle": cycle, "repetition": repetition, "accuracy": (predicted == part["camp"]).mean()})
repeats = pd.DataFrame(rows)
repeats.to_csv(RESULTS_DIR / "h4b_check_repeats.csv", index=False)

In [ ]:
by_model = repeats.groupby(["model", "cycle"])["accuracy"].agg(mean="mean", p2_5=lambda values: values.quantile(0.025), p97_5=lambda values: values.quantile(0.975))
by_model.round(4)

## 6. Różnica 2024 − 2016

In [ ]:
wide = repeats.pivot_table(index=["model", "repetition"], columns="cycle", values="accuracy")
wide["diff"] = wide[2024] - wide[2016]
difference = wide.groupby("model")["diff"].agg(mean="mean", p2_5=lambda values: values.quantile(0.025),
                                               p97_5=lambda values: values.quantile(0.975),
                                               share_2024_higher=lambda values: (values > 0).mean())
difference.to_csv(RESULTS_DIR / "h4b_check_difference.csv")
difference.round(4)

## 7. Poziom przypadku (przemieszane etykiety obozów)

In [ ]:
rows = []
for repetition in tqdm(range(SHUFFLE_REPEATS)):
    for cycle in CHECK_CYCLES:
        part = data[data["cycle"] == cycle].groupby(CELLS).sample(per_cell, random_state=repetition)
        shuffled = part["camp"].sample(frac=1, random_state=repetition).to_numpy()
        predicted = cross_val_predict(main_model, part["text"], shuffled, cv=GroupKFold(n_splits=5), groups=part["link_id"], n_jobs=-1)
        rows.append({"cycle": cycle, "repetition": repetition, "accuracy": (predicted == shuffled).mean()})
shuffled_accuracy = pd.DataFrame(rows)
shuffled_accuracy.groupby("cycle")["accuracy"].mean().round(4)

## 8. Krzywa uczenia (model główny)

In [ ]:
rows = []
for camp_size in tqdm([size for size in LEARNING_SIZES if size < 4 * per_cell] + [4 * per_cell]):
    for repetition in range(LEARNING_REPEATS):
        accuracies = {}
        for cycle in CHECK_CYCLES:
            part = data[data["cycle"] == cycle].groupby(CELLS).sample(camp_size // 4, random_state=repetition)
            predicted = cross_val_predict(main_model, part["text"], part["camp"], cv=GroupKFold(n_splits=5), groups=part["link_id"], n_jobs=-1)
            accuracies[cycle] = (predicted == part["camp"]).mean()
        rows.append({"camp_size": camp_size, "repetition": repetition, "acc_2016": accuracies[2016], "acc_2024": accuracies[2024]})
learning = pd.DataFrame(rows)
learning["diff"] = learning["acc_2024"] - learning["acc_2016"]
learning_summary = learning.groupby("camp_size").agg(acc_2016=("acc_2016", "mean"), acc_2024=("acc_2024", "mean"),
                                                     diff=("diff", "mean"), diff_p2_5=("diff", lambda values: values.quantile(0.025)),
                                                     diff_p97_5=("diff", lambda values: values.quantile(0.975)))
learning_summary["detectable_difference"] = (learning_summary["diff_p97_5"] - learning_summary["diff_p2_5"]) / 2
learning_summary.to_csv(RESULTS_DIR / "h4b_check_learning.csv")
learning_summary.round(4)

## 9. Słowa wyróżniające (model główny dopasowany na całej próbie cyklu)

In [ ]:
LEAK_HINTS = ["democrats", "liberal", "republican", "conservative", "flair", "mods", "removed", "rule"]
rows = []
for cycle in CHECK_CYCLES:
    part = data[data["cycle"] == cycle].groupby(CELLS).sample(per_cell, random_state=0)
    main_model.fit(part["text"], part["camp"])
    assert list(main_model.classes_) == ["dem", "rep"]
    words = main_model[0].get_feature_names_out()
    coefficients = main_model[1].coef_[0]
    order = coefficients.argsort()
    for camp, indices in (("dem", order[:20]), ("rep", order[::-1][:20])):
        for rank, index in enumerate(indices, start=1):
            rows.append({"cycle": cycle, "camp": camp, "rank": rank, "word": words[index], "coef": coefficients[index],
                         "leak_hint": words[index] in LEAK_HINTS})
top_words = pd.DataFrame(rows)
top_words.to_csv(RESULTS_DIR / "h4b_check_top_words.csv", index=False)
top_words.pivot_table(index=["cycle", "rank"], columns="camp", values="word", aggfunc="first").head(40)